#Experimentation Case Study: Trial vs Direct Sale (A/A + A/B + Segmentation + Business Impact)

#Business Problem

###The company is evaluating whether introducing a Trial-First onboarding experience can improve early user activation and overall business outcomes.

###The current Direct Sale approach may create friction for users who are hesitant to commit upfront, which can impact both activation and conversion. By offering a trial experience, the company aims to reduce this friction, enabling users to explore the product before making a purchase decision.

###The key objective is to determine whether the Trial-First strategy leads to a meaningful improvement in early activation while maintaining or improving downstream business metrics such as revenue, refund rates, and support load. Additionally, the company wants to assess whether this approach helps sales representatives by reducing friction in the conversion process.

###2.
### Experiment Design Assumptions

###Primary Metric: Early activation within 7 days  
###Baseline Activation Rate: ~15%  
###Minimum Detectable Effect: 2 percentage points  
###Significance Level: 5%  
###Power: 80%  

###A/A validation
###A/B testing
###Metrics:
###Primary Metrics → Activation
###Guardrail Metrics → Refunds, Support
###Business Outcomes → Revenue

In [72]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv("/content/trial_vs_direct_sale_experiment_dataset.csv")

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   user_id              200000 non-null  int64  
 1   experiment_stage     200000 non-null  object 
 2   variant              200000 non-null  object 
 3   offer_strategy       200000 non-null  object 
 4   device_type          200000 non-null  object 
 5   location             200000 non-null  object 
 6   user_type            200000 non-null  object 
 7   sessions_14d         200000 non-null  int64  
 8   early_activation_7d  200000 non-null  int64  
 9   refund_30d           200000 non-null  int64  
 10  support_ticket_14d   200000 non-null  int64  
 11  net_revenue_30d      200000 non-null  float64
 12  contamination_flag   200000 non-null  int64  
dtypes: float64(1), int64(6), object(6)
memory usage: 19.8+ MB


In [73]:
ab_df = df[df['experiment_stage'] == 'AB']

In [74]:
# User Mix Balance Check by Device Type
balance_snapshot = ab_df.groupby(['device_type', 'variant']).agg(
    users=('user_id', 'count')
).reset_index()

balance_snapshot

,device_type,variant,users
0,Desktop,Control,20562
1,Desktop,Treatment,20726
2,Mobile,Control,54713
3,Mobile,Treatment,54416
4,Tablet,Control,4799
5,Tablet,Treatment,4835


###Before interpreting the A/B test results, I checked whether Control and Treatment had similar user composition across device types. This helps confirm that the treatment effect is not being driven by one group having more mobile, desktop, or tablet users.

###A/A Test Validation

In [75]:
# Filter A/A data
aa_df = df[df['experiment_stage'] == 'AA']

# -----------------------------
# 1. Sample Ratio Mismatch (SRM) Check
# -----------------------------
from scipy.stats import chisquare

# Count users in each group
counts = aa_df['variant'].value_counts().sort_index()

# Expected split (50/50)
expected = [len(aa_df) / 2, len(aa_df) / 2]

# Chi-square test
chi_stat, p_value = chisquare(f_obs=counts, f_exp=expected)

print("----- SRM CHECK (Traffic Split) -----")
print("Observed counts:\n", counts)
print("\nExpected counts:\n", expected)
print("\nChi-square statistic:", round(chi_stat, 4))
print("P-value:", p_value)

if p_value > 0.05:
    print("✅ No Sample Ratio Mismatch: Traffic split is valid")
else:
    print("❌ Sample Ratio Mismatch detected: Randomization issue")


# -----------------------------
# 2. Metric Parity Check (A/A Test)
# -----------------------------
from scipy.stats import ttest_ind

# Split groups
control = aa_df[aa_df['variant'] == 'Control']['early_activation_7d']
treatment = aa_df[aa_df['variant'] == 'Treatment']['early_activation_7d']

# T-test
t_stat, metric_p_value = ttest_ind(control, treatment)

print("\n----- METRIC PARITY CHECK -----")
print("Control Mean:", round(control.mean(), 4))
print("Treatment Mean:", round(treatment.mean(), 4))
print("T-statistic:", round(t_stat, 4))
print("P-value:", metric_p_value)

if metric_p_value > 0.05:
    print("✅ No significant difference: Metrics are consistent across groups")
else:
    print("❌ Significant difference detected: Possible tracking issue")


# -----------------------------
# 3. Final Conclusion
# -----------------------------
print("\n----- FINAL A/A VALIDATION RESULT -----")

if p_value > 0.05 and metric_p_value > 0.05:
    print("✅ A/A test passed: Experiment setup is valid and unbiased")
else:
    print("❌ A/A test failed: Investigate randomization or tracking issues")

----- SRM CHECK (Traffic Split) -----
Observed counts:
 variant
Control      20033
Treatment    19916
Name: count, dtype: int64

Expected counts:
 [19974.5, 19974.5]

Chi-square statistic: 0.3427
P-value: 0.558296741447877
✅ No Sample Ratio Mismatch: Traffic split is valid

----- METRIC PARITY CHECK -----
Control Mean: 0.1499
Treatment Mean: 0.1538
T-statistic: -1.0981
P-value: 0.27214882340435886
✅ No significant difference: Metrics are consistent across groups

----- FINAL A/A VALIDATION RESULT -----
✅ A/A test passed: Experiment setup is valid and unbiased


#A chi-square test was conducted to validate whether users were evenly distributed across control and treatment groups. The observed distribution closely matched the expected 50/50 split (p = 0.558), indicating no sample ratio mismatch. This confirms that the randomization process is functioning correctly and does not introduce bias into the experiment

In [76]:
ab_df.groupby('variant')['contamination_flag'].mean() * 100

,contamination_flag
variant,
Control,5.371282
Treatment,5.330282


#Experiment results snapshot

#A/B Statistical Test with early_activation_7d as a Primary Metric)

In [77]:
conv = ab_df.groupby('variant')['early_activation_7d'].sum().values
n = ab_df.groupby('variant')['early_activation_7d'].count().values

stat, p_value = proportions_ztest(conv, n)
print("P-value:", p_value)

P-value: 8.416875178331508e-58


#We used a two-proportion z-test because early activation is a binary conversion metric. This was appropriate for testing whether the activation rate differed between Control and Treatment.

###We reject the null hypothesis

###The difference in activation between Trial-First and Direct Sale is statistically significant

##Lift Calculation

In [78]:
control_rate = ab_df[ab_df['variant']=='Control']['early_activation_7d'].mean()
treatment_rate = ab_df[ab_df['variant']=='Treatment']['early_activation_7d'].mean()

print(control_rate, treatment_rate)

0.15068561580538004 0.1804643835102592


In [79]:
absolute_lift = (treatment_rate - control_rate) * 100   # percentage points
relative_lift = ((treatment_rate - control_rate) / control_rate) * 100  # %

print(f"Absolute Lift: {absolute_lift:.2f} percentage points")
print(f"Relative Lift: {relative_lift:.2f}%")

Absolute Lift: 2.98 percentage points
Relative Lift: 19.76%


#Lets compare with Guardrail Metrics

In [80]:
ab_df.groupby('variant')[['refund_30d','support_ticket_14d']].mean() * 100

,refund_30d,support_ticket_14d
variant,,
Control,6.194270,9.639833
Treatment,7.100791,10.470510


#Lift for Guardrails

In [81]:
guardrail = ab_df.groupby('variant')[['refund_30d','support_ticket_14d']].mean().T

guardrail_pct = (guardrail * 100).round(2)

guardrail_pct['Lift (%)'] = (
    (guardrail['Treatment'] - guardrail['Control']) / guardrail['Control'] * 100
).round(2)

guardrail_pct

variant,Control,Treatment,Lift (%)
refund_30d,6.19,7.10,14.63
support_ticket_14d,9.64,10.47,8.62


###We do see a bump in support tickets, but it’s relatively small compared to the big gain in activation. So the question becomes, are we okay handling a bit more support if it means getting a lot more users engaged? In most cases, that’s a trade-off worth considering, especially for valuable user segments.

#Revenue

In [82]:
ab_df.groupby('variant')['net_revenue_30d'].mean()

,net_revenue_30d
variant,
Control,624.826391
Treatment,596.244857


#Revenue Lift

In [83]:
ab_test = df[df['experiment_stage'] == 'AB']

rev = ab_test.groupby('variant')['net_revenue_30d'].mean()

rev_lift = (
    (rev['Treatment'] - rev['Control']) / rev['Control'] * 100
)

print("Average Revenue by Variant:")
print(rev)

print("Revenue Lift (%):", round(rev_lift, 2))

Average Revenue by Variant:
variant
Control      624.826391
Treatment    596.244857
Name: net_revenue_30d, dtype: float64
Revenue Lift (%): -4.57


In [84]:
rev_lift = (
    (rev['Treatment'] - rev['Control']) / rev['Control'] * 100
)

print("Revenue Lift (%):", round(rev_lift, 2))

Revenue Lift (%): -4.57


#The Trial-First approach is effective at getting more users to try the product, but many of these users don’t convert into paying or valuable customers. As a result, overall revenue drops despite higher activation.

#Confidence Intervals

In [85]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1=ab_df[ab_df['variant']=='Treatment']['early_activation_7d'].sum(),
    nobs1=len(ab_df[ab_df['variant']=='Treatment']),
    count2=ab_df[ab_df['variant']=='Control']['early_activation_7d'].sum(),
    nobs2=len(ab_df[ab_df['variant']=='Control']),
    method='score'
)

print(ci_low, ci_high)

0.026142801234127697 0.03341398160267702


###We are 95% confident that the Trial feature improves activation by between 2.6% and 3.3%

#Segmentation

#By Device

In [86]:
device_seg = ab_df.groupby(['device_type','variant'])['early_activation_7d'].mean().unstack()

# Convert to %
device_seg_pct = (device_seg * 100).round(2)

# Add Lift %
device_seg_pct['Lift (%)'] = (
    (device_seg['Treatment'] - device_seg['Control']) / device_seg['Control'] * 100
).round(2)

device_seg_pct

variant,Control,Treatment,Lift (%)
device_type,,,
Desktop,14.09,16.17,14.76
Mobile,15.45,18.88,22.20
Tablet,14.94,16.75,12.13


#Trial-First improves early activation across all device types, with the strongest impact observed on mobile users (+22.2% lift), followed by desktop (+14.8%) and tablet (+12.1%).

#This suggest mobile users are more responsive to trial first experience

#By User Type

In [87]:
# Activation
activation_seg = ab_df.groupby(['user_type','variant'])['early_activation_7d'].mean().unstack()

# Revenue
revenue_seg = ab_df.groupby(['user_type','variant'])['net_revenue_30d'].mean().unstack()

# Combine into one table
insight_table = pd.DataFrame({
    'Activation_Control (%)': (activation_seg['Control'] * 100).round(2),
    'Activation_Treatment (%)': (activation_seg['Treatment'] * 100).round(2),
    'Activation_Lift (%)': ((activation_seg['Treatment'] - activation_seg['Control']) / activation_seg['Control'] * 100).round(2),

    'Revenue_Control': revenue_seg['Control'].round(2),
    'Revenue_Treatment': revenue_seg['Treatment'].round(2),
    'Revenue_Lift (%)': ((revenue_seg['Treatment'] - revenue_seg['Control']) / revenue_seg['Control'] * 100).round(2)
})

insight_table

,Activation_Control (%),Activation_Treatment (%),Activation_Lift (%),Revenue_Control,Revenue_Treatment,Revenue_Lift (%)
user_type,,,,,,
New,14.48,17.65,21.94,582.48,559.07,-4.02
Returning,17.14,19.43,13.33,773.77,727.60,-5.97


###The Trial-First strategy should not be applied uniformly.
###It is effective for new users but may harm monetization among returning users. A segmented rollout is recommended.

#Location

In [88]:
loc_seg = ab_df.groupby(['location','variant'])['early_activation_7d'].mean().unstack()

# Convert to %
loc_seg_pct = (loc_seg * 100).round(2)

# Add Lift %
loc_seg_pct['Lift (%)'] = (
    (loc_seg['Treatment'] - loc_seg['Control']) / loc_seg['Control'] * 100
).round(2)

# Optional pretty format
loc_seg_pct['Lift (%)'] = loc_seg_pct['Lift (%)'].apply(lambda x: f"+{x}" if x > 0 else x)

loc_seg_pct

variant,Control,Treatment,Lift (%)
location,,,
East India,14.44,16.54,+14.57
North India,14.75,17.84,+20.94
South India,15.31,18.93,+23.67
West India,15.49,18.04,+16.42


#Trial-First improves activation across all regions, with the strongest lift observed in South India (+23.67%), followed by North India (+20.94%).

#Final Summary Readout

In [89]:
summary = ab_df.groupby('variant').agg({
    'early_activation_7d': 'mean',
    'refund_30d': 'mean',
    'support_ticket_14d': 'mean',
    'net_revenue_30d': 'mean'
})

# Convert to %
summary_pct = summary.copy()
summary_pct[['early_activation_7d','refund_30d','support_ticket_14d']] *= 100
summary_pct = summary_pct.round(2)

# Transpose for clean format
summary_t = summary_pct.T

# Add Lift %
summary_t['Lift (%)'] = (
    (summary_t['Treatment'] - summary_t['Control']) / summary_t['Control'] * 100
).round(2)

summary_t

variant,Control,Treatment,Lift (%)
early_activation_7d,15.07,18.05,19.77
refund_30d,6.19,7.10,14.70
support_ticket_14d,9.64,10.47,8.61
net_revenue_30d,624.83,596.24,-4.58


#Final Interpretation

###The Trial-First approach increases early user engagement, indicating reduced onboarding friction. However, this improvement comes with a decline in revenue and signals of lower user quality, including higher refund rates and increased support requests.

###This suggests a tradeoff between engagement and monetization. Rather than a full rollout, a targeted strategy is more appropriate. The feature performs best for segments such as new users, where the activation gains outweigh potential downsides.

###A selective rollout allows us to capture the benefits of increased engagement while minimizing revenue risk from high-intent users.

#Experiment Risks & Validity Checks

###Contamination

###There is a risk of contamination if users are exposed to both control and treatment groups especially in the case of multi device or cross session environments

###Sample Ratio Mismatch

###If the distribution is unequal , group can bias the results

###We performed an SRM check to ensure that users were evenly distributed between groups

###Metric Misalignment

###Optimizing one metric harms others
###So We basically added primary gaurdrails and revenue to see the overall metrics

#### Guardrail Metrics Interpretation

###For this analysis, I treated revenue, refunds, and support tickets as business-critical guardrail metrics. While I did not run formal statistical significance tests on every guardrail metric, the directional movement itself is important because the business has low tolerance for downside risk in these areas.

###Even if the increases in refunds or support tickets are not statistically significant, they still create operational and financial risk. Therefore, I would not recommend a full rollout based only on the activation lift. Instead, I would recommend a targeted rollout or a follow-up experiment to validate whether Trial-First can improve activation without negatively affecting revenue quality, refund behavior, or support burden.